# 本文件展示了LGflow与torch在链式求导以及梯度更新时的对比

In [1]:
from torch import nn as t_nn
from torch import optim as t_optim
import torch

In [2]:
from LG_flow import nn as l_nn
from LG_flow import optim as l_optim
import LG_flow

# 一. 创建模型

## 1.1 创建torch模型
    创建一个两层全连接的torch模型。

In [3]:
class TNet(t_nn.Module):
    def __init__(self):
        super().__init__()

        self.fc1 = t_nn.Linear(5, 4)
        self.norm = t_nn.LayerNorm(4) # 不要问为什么是LayerNorm，这里只是测试不同。
        self.act1 = t_nn.ReLU()
        self.fc2 = t_nn.Linear(4, 3)
        self.act2 = t_nn.Softmax()
        
    def forward(self, x):
        x = self.fc1(x)
        x = self.norm(x)
        x = self.act1(x)
        x = self.fc2(x)
        x = self.act2(x)
        return x

t_net = TNet()

In [4]:
t_net

TNet(
  (fc1): Linear(in_features=5, out_features=4, bias=True)
  (norm): LayerNorm((4,), eps=1e-05, elementwise_affine=True)
  (act1): ReLU()
  (fc2): Linear(in_features=4, out_features=3, bias=True)
  (act2): Softmax(dim=None)
)

## 1.2 创建LGflow模型
    通过LGflow，创建一个结构相同的模型。

In [5]:
class LNet(l_nn.Module):
    def __init__(self):
        super().__init__()

        self.fc1 = l_nn.Linear(5, 4)
        self.norm = l_nn.LayerNorm(4)
        self.act1 = l_nn.ReLU()
        self.fc2 = l_nn.Linear(4, 3)
        self.act2 = l_nn.Softmax()

    def forward(self, x):
        x = self.fc1(x)
        x = self.norm(x)
        x = self.act1(x)
        x = self.fc2(x)
        x = self.act2(x)
        return x

l_net = LNet()

In [6]:
print(l_net)

## 1.3 使LGflow模型的参数与torch模型参数相同

In [7]:
l_net.fc1.weights = l_nn.Parameter(t_net.fc1.weight.data.numpy())
l_net.fc1.bias = l_nn.Parameter(t_net.fc1.bias.data.numpy())
l_net.fc2.weights = l_nn.Parameter(t_net.fc2.weight.data.numpy())
l_net.fc2.bias = l_nn.Parameter(t_net.fc2.bias.data.numpy())

# 二. 创建输入

In [8]:
t_x = torch.randn(2, 5)    # input for torch
t_t = torch.tensor([[1, 0, 0], [0, 1, 0]], dtype=torch.float32)    # target for torch

In [9]:
t_x

tensor([[-1.6827, -0.3180,  2.1436,  1.1439,  1.7261],
        [-1.0065,  2.1189, -1.9873, -0.3857, -0.1534]])

In [10]:
l_x = LG_flow.Tensor(t_x.data.numpy())    # input for LGflow
l_t = LG_flow.Tensor([[1, 0, 0], [0, 1, 0]])    # target for LGflow

In [11]:
print(l_x)

(Tensor shape=(2, 5) dtype=float32 required_grad=False grad_fn=None 
[[-1.6827017  -0.31801653  2.1436203   1.143921    1.7261252 ]
 [-1.0064574   2.118916   -1.987266   -0.3857404  -0.15342954]]
)


# 三. 创建损失函数

In [12]:
t_loss_fn = t_nn.CrossEntropyLoss(reduction='sum')

In [13]:
l_loss_fn = l_nn.CrossEntropyLoss(reduction='sum')

# 四. 创建优化器
    这里使用一个较大的学习率，使参数的更新幅度更大

In [14]:
t_optimizer = t_optim.SGD(t_net.parameters(), lr=0.1)

In [15]:
l_optimizer = l_optim.SGD(l_net.parameters(), lr=0.1)

# 五. 反向传播更新模型参数

    以epoch次反向更新为例，展示了LGflow与torch在链式求导与梯度更新方面具有一致性。

In [16]:
epochs = 20

In [17]:
for epoch in range(epochs):
    print('-'*50 + f'epoch: {epoch}' + '-'*50)
    
    # torch forward
    t_y = t_net(t_x)
    print('t_y: ', t_y)

    # LGflow forward
    l_y = l_net(l_x)
    print('l_y: ', l_y)
    print()
    
    # torch loss
    t_loss = t_loss_fn(t_y, t_t)
    print('t_loss: ', t_loss)

    # LGflow loss
    l_loss = l_loss_fn(l_y, l_t)
    print('l_loss: ', l_loss)
    print()

    # torch backward
    t_optimizer.zero_grad()
    t_loss.backward()
    t_optimizer.step()
    
    # LGflow backward
    l_optimizer.zero_grad()
    l_loss.backward()
    l_optimizer.step()
    
    # check parameters
    ## torch parameters
    for param in t_net.parameters():
        print('torch model paramenters 0 value: ')
        print(param)
        print('torch model paramenters 0 grad: ')
        print(param.grad)
        break
        
    print()
    
    ## LGflow parameters
    for name, param in l_net.parameters().items():
        print('LGflow model paramenters 0 value: ')
        print(param)
        print('LGflow model paramenters 0 grad: ')
        print(param.grad)
        break


--------------------------------------------------epoch: 0--------------------------------------------------
t_y:  tensor([[0.3606, 0.2921, 0.3473],
        [0.2006, 0.5175, 0.2819]], grad_fn=<SoftmaxBackward0>)
l_y:  (Tensor shape=(2, 3) dtype=float32 required_grad=False grad_fn=<LG_flow.Math_op.DIV_WITH_TENSOR object at 0x7a6b49273490> 
[[0.3606413  0.29206336 0.34729534]
 [0.200571   0.51750284 0.2819262 ]]
)

t_loss:  tensor(1.9954, grad_fn=<NegBackward0>)
l_loss:  (Tensor shape=() dtype=float32 required_grad=False grad_fn=<LG_flow.Math_op.SUM object at 0x7a6b49273940> 
1.9954040050506592
)

torch model paramenters 0 value: 
Parameter containing:
tensor([[ 0.2091, -0.3078,  0.4574, -0.0850,  0.0705],
        [ 0.4286,  0.3458,  0.0726, -0.3414,  0.2615],
        [-0.3222, -0.3026,  0.3626, -0.3326, -0.3475],
        [ 0.0939, -0.2095, -0.3689,  0.3077, -0.0906]], requires_grad=True)
torch model paramenters 0 grad: 
tensor([[ 0.0790,  0.0173, -0.1040, -0.0548, -0.0823],
        [ 0.

/home/lg/anaconda3/envs/lgflow_env/lib/python3.8/site-packages/torch/nn/modules/module.py:1553: UserWarning: Implicit dimension choice for softmax has been deprecated. Change the call to include dim=X as an argument.
  return self._call_impl(*args, **kwargs)
